## Group: [153] Italienerne
#### Authors:
- Riccardo Mazzoleni    152564
- Simone Tolledi        151684
- Samrath Singh Chhabra 152583

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import pathlib
import os
import json
import random
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

## Data and Preprocessing

In [ ]:
cd = pathlib.Path.cwd()
PROJECT_ROOT = cd.parent

def load_data():
    
    try:
        receivals_file = 'receivals.csv'
        po_file        = 'purchase_orders.csv'
        materials_file = 'materials.csv'
        mapping_file   = 'prediction_mapping.csv'
        
        receivals          = pd.read_csv(receivals_file)
        purchase_orders    = pd.read_csv(po_file)
        materials          = pd.read_csv(materials_file)
        prediction_mapping = pd.read_csv(mapping_file)

        receivals['date_arrival']                 = pd.to_datetime(receivals['date_arrival'], utc=True, errors='coerce').dt.date
        purchase_orders['delivery_date']          = pd.to_datetime(purchase_orders['delivery_date'], utc=True, errors='coerce').dt.date
        prediction_mapping['forecast_start_date'] = pd.to_datetime(prediction_mapping['forecast_start_date'], errors='coerce').dt.date
        prediction_mapping['forecast_end_date']   = pd.to_datetime(prediction_mapping['forecast_end_date'], errors='coerce').dt.date

        receivals       = receivals.dropna(subset=['date_arrival', 'rm_id', 'net_weight'])
        purchase_orders = purchase_orders.dropna(subset=['delivery_date', 'product_id', 'quantity'])

        receivals = receivals.loc[receivals['net_weight'] > 0].copy()
        purchase_orders = purchase_orders.loc[purchase_orders['quantity'] > 0].copy()

        conversion_to_kg = 0.453592
        pounds_mask = (purchase_orders['unit_id'] == 43)
        purchase_orders.loc[pounds_mask, 'quantity'] = purchase_orders.loc[pounds_mask, 'quantity'] * conversion_to_kg
        purchase_orders.loc[pounds_mask, 'unit'] = 'kg'
        purchase_orders.loc[pounds_mask, 'unit_id'] = 40

        return receivals, purchase_orders, materials, prediction_mapping

    except FileNotFoundError as e:
        print(f"Error: File not found: {e}")
        return None, None, None, None

# Aggregates receivals and purchase orders to daily totals per rm_id.
def aggregate_daily_weight(receivals, purchase_orders, materials):
    
    daily_receivals = receivals.groupby(['rm_id', 'date_arrival']).net_weight.sum().reset_index()
    rm_product_map = materials[['rm_id', 'product_id']].drop_duplicates().dropna()
    po_with_rm = pd.merge(purchase_orders, rm_product_map, on='product_id', how='left').dropna(subset=['rm_id'])
    daily_po = po_with_rm.groupby(['rm_id', 'delivery_date']).quantity.sum().reset_index()

    # Types
    daily_receivals['rm_id'] = daily_receivals['rm_id'].astype(int)
    daily_po['rm_id'] = daily_po['rm_id'].astype(int)
    return daily_receivals, daily_po

# classic Croston method for intermittent demand forecasting.
def _croston_forecast(y, h=151, alpha=0.1):
    """Classic Croston for intermittent daily series (nonnegative).
    y: 1D array of daily net_weight (may contain many zeros). Returns h-step mean forecast per day."""
    y = np.asarray(y, dtype=float)
    demand = y[y > 0.0]
    if demand.size == 0:
        return np.zeros(h, dtype=float)
    intervals = np.diff(np.concatenate(([0], np.where(y > 0)[0])))
    z = demand[0]; p = intervals[0] if intervals.size > 0 else 1.0
    for k in range(1, demand.size):
        z = alpha * demand[k]   + (1 - alpha) * z
    for k in range(1, intervals.size):
        p = alpha * intervals[k] + (1 - alpha) * p
    rate = z / max(p, 1e-6)  # expected demand per day
    return np.full(h, rate, dtype=float)

# Builds a daily baseline forecast for each rm_id using Croston’s method.
def build_intermittent_baseline(daily_receivals, start_date, end_date, method="croston", alpha=0.1):
    
    horizon_days = (end_date - start_date).days + 1
    dr = daily_receivals.copy()
    dr['date_arrival'] = pd.to_datetime(dr['date_arrival'])
    result = []
    for rm, g in dr.groupby('rm_id'):
        g = g.set_index('date_arrival').sort_index()
        # reindex daily since min date
        s = g['net_weight'].asfreq('D', fill_value=0.0)
        # Use last N years only (e.g. 3y) to avoid ancient regime
        cutoff = pd.Timestamp(start_date) - pd.Timedelta(days=365*3)
        s_hist = s[s.index < pd.Timestamp(start_date)]
        if s_hist.index.min() is None:
            s_hist = s[s.index < pd.Timestamp(start_date)]
        s_hist = s_hist[s_hist.index >= cutoff]
        if s_hist.empty:
            fc = np.zeros(horizon_days, dtype=float)
        else:
            if method == "croston":
                fc = _croston_forecast(s_hist.values, h=horizon_days, alpha=alpha)
            else:
                fc = _croston_forecast(s_hist.values, h=horizon_days, alpha=alpha)  # fallback
        dates = pd.date_range(start_date, end_date, freq='D')
        result.append(pd.DataFrame({'rm_id': rm, 'date': dates, 'baseline_daily_fc': fc}))
    return pd.concat(result, ignore_index=True)

# Aggregates the Croston daily baseline forecast over each forecasting window.
def sum_baseline_over_windows(features_df, baseline_df):
    """
    Given windows [forecast_start_date, forecast_end_date], sum baseline_daily_fc within each window per rm_id.
    Returns Series aligned to features_df.index with name 'baseline_sum_in_window'.
    """
    idx = features_df.index
    rm = features_df['rm_id'].values
    starts = pd.to_datetime(features_df['forecast_start_date']).values.astype('datetime64[D]')
    ends   = pd.to_datetime(features_df['forecast_end_date']).values.astype('datetime64[D]')
    # build map rm_id -> arrays for vectorized summation
    base_map = {}
    for k, g in baseline_df.groupby('rm_id'):
        d = pd.to_datetime(g['date']).values.astype('datetime64[D]')
        v = g['baseline_daily_fc'].values.astype(float)
        base_map[k] = (d, np.cumsum(v))
    out = np.zeros(len(features_df), dtype=float)
    for i in range(len(features_df)):
        tup = base_map.get(rm[i])
        if tup is None: 
            continue
        d, c = tup
        l = np.searchsorted(d, starts[i], side='left')
        r = np.searchsorted(d, ends[i],   side='right') - 1
        if r >= l and l < len(c):
            left = c[l-1] if l > 0 else 0.0
            out[i] = c[r] - left
    return pd.Series(out, index=idx, name='baseline_sum_in_window')


## Feature engineering

In [ ]:
def create_features(df, daily_receivals, daily_po, receivals_full=None, purchase_orders_full=None):
    
    print(f"Creating ENHANCED features for {len(df)} rows...")

    original_index_name = df.index.name
    features = df.reset_index()
    if 'index' not in features.columns and original_index_name is not None:
        features = features.rename(columns={original_index_name: 'index'})

    # ---- Window geometry ----
    features['window_length'] = (
        features['forecast_end_date'] - features['forecast_start_date']
    ).apply(lambda x: x.days + 1).astype(np.int16)

    # Calendar anchors (END date)
    features['end_month'] = features['forecast_end_date'].apply(lambda x: x.month).astype(np.int16)
    features['end_dayofweek'] = features['forecast_end_date'].apply(lambda x: x.weekday()).astype(np.int16)

    # Cyclical month encoding
    features['month_sin'] = np.sin(2 * np.pi * features['end_month'] / 12).astype(np.float32)
    features['month_cos'] = np.cos(2 * np.pi * features['end_month'] / 12).astype(np.float32)

    # Business days / weekends in window
    starts_np = pd.to_datetime(features["forecast_start_date"]).values.astype('datetime64[D]')
    ends_np   = pd.to_datetime(features["forecast_end_date"]).values.astype('datetime64[D]')
    features["business_days_in_window"] = [
        np.busday_count(s, e + np.timedelta64(1, 'D')) for s, e in zip(starts_np, ends_np)
    ]
    features["business_days_in_window"] = features["business_days_in_window"].astype(np.int16)
    features["weekends_in_window"] = (features["window_length"] - features["business_days_in_window"]).astype(np.int16)

    # Distance to month end
    end_ts = pd.to_datetime(features["forecast_end_date"])
    features["end_days_to_month_end"] = (end_ts.dt.days_in_month - end_ts.dt.day).astype(np.int16)

    # ---- PO in window (per-rm, RAM-safe) ----
    print("  Computing po_in_window feature (per-rm)...")
    pp = daily_po.copy()
    pp['delivery_date'] = pd.to_datetime(pp['delivery_date'], errors='coerce')
    po_by_rm = {rm: grp.sort_values('delivery_date').reset_index(drop=True)
                for rm, grp in pp.groupby('rm_id')} if not pp.empty else {}
    po_in_window_vals = np.zeros(len(features), dtype=float)

    for rm, rows in features.groupby('rm_id'):
        row_positions = rows.index.values
        po_grp = po_by_rm.get(rm)
        if po_grp is None or po_grp.empty:
            continue
        po_dates = pd.to_datetime(po_grp['delivery_date']).values.astype('datetime64[D]')
        po_qty = po_grp['quantity'].values.astype(float)
        po_cum = np.cumsum(po_qty)

        starts = pd.to_datetime(rows['forecast_start_date']).values.astype('datetime64[D]')
        ends   = pd.to_datetime(rows['forecast_end_date']).values.astype('datetime64[D]')
        left_pos  = np.searchsorted(po_dates, starts, side='left')
        right_pos = np.searchsorted(po_dates, ends,   side='right') - 1

        sums = np.zeros(len(row_positions), dtype=float)
        for i, (l, r) in enumerate(zip(left_pos, right_pos)):
            if r >= l and l < len(po_cum):
                left_cum = po_cum[l-1] if l > 0 else 0.0
                sums[i] = po_cum[r] - left_cum
        po_in_window_vals[row_positions] = sums

    features['po_in_window'] = po_in_window_vals
    features['daily_avg_po_in_window'] = (
        features['po_in_window'] / features['window_length']
    ).replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)

    # ---- Historical aggregates & rolling stats ----
    print("  Creating historical features...")
    hist_agg_list = []
    rec_horizons = [7, 30, 90]
    po_horizons  = [7, 30, 90]
    rec_rolling_windows = [30, 90]

    dr = daily_receivals.copy()
    dr['date_arrival'] = pd.to_datetime(dr['date_arrival'], errors='coerce')

    for start_date in features['forecast_start_date'].unique():
        ref_date = pd.to_datetime(start_date) - pd.Timedelta(days=1)
        rec_frames, po_frames = [], []

        # Receivals sums
        for h in rec_horizons:
            hist_start = ref_date - pd.Timedelta(days=(h - 1))
            hist_rec = dr[(dr['date_arrival'] >= hist_start) & (dr['date_arrival'] <= ref_date)]
            
            rec_agg = hist_rec.groupby('rm_id')['net_weight'].sum()
            rec_agg.name = f'hist_rec_{h}d'
            rec_frames.append(rec_agg)

        # PO sums
        for h in po_horizons:
            hist_start = ref_date - pd.Timedelta(days=(h - 1))
            hist_po = daily_po[
                (pd.to_datetime(daily_po['delivery_date']) >= hist_start) &
                (pd.to_datetime(daily_po['delivery_date']) <= ref_date)
            ]

            po_agg = hist_po.groupby('rm_id')['quantity'].sum()
            po_agg.name = f'hist_po_{h}d'
            po_frames.append(po_agg)

        # Rolling mean / std / trend + counts
        for window in rec_rolling_windows:
            hist_start = ref_date - pd.Timedelta(days=(window - 1))
            hist_rec = dr[(dr['date_arrival'] >= hist_start) & (dr['date_arrival'] <= ref_date)]
            
            rec_mean = hist_rec.groupby('rm_id')['net_weight'].mean()
            rec_mean.name = f'rec_mean_{window}d'
            
            rec_std = hist_rec.groupby('rm_id')['net_weight'].std()
            rec_std.name = f'rec_std_{window}d'
            
            def _trend(g):
                if len(g) < 2:
                    return 0.0
                t = (g['date_arrival'] - g['date_arrival'].min()).dt.days.values
                if np.unique(t).size < 2:
                    return 0.0
                return float(np.polyfit(t, g['net_weight'].values, 1)[0])
            
            # fix: constrains the result to a Series
            if hist_rec.empty:
                rec_trend = pd.Series(dtype=float, name=f'rec_trend_{window}d')
                rec_trend.index.name = 'rm_id'
            else:
                trend_result = hist_rec.groupby('rm_id').apply(_trend)
                
                if isinstance(trend_result, pd.DataFrame):
                    rec_trend = trend_result.iloc[:, 0]  
                    rec_trend.name = f'rec_trend_{window}d'
                else:
                    rec_trend = trend_result
                    rec_trend.name = f'rec_trend_{window}d'
            
            rec_cnt = hist_rec.groupby('rm_id')['date_arrival'].nunique()
            rec_cnt.name = f'rec_count_{window}d'
            
            rec_frames.extend([rec_mean, rec_std, rec_trend, rec_cnt])
        
        all_aggs = pd.concat(rec_frames + po_frames, axis=1)
        all_aggs = all_aggs.fillna(0).reset_index()
        
        all_aggs['forecast_start_date'] = start_date
        hist_agg_list.append(all_aggs)

    print(f"\n=== DEBUG per start_date={start_date} ===")
    for i, frame in enumerate(rec_frames + po_frames):
        if isinstance(frame, pd.DataFrame):
            print(f"Frame {i}: DataFrame con colonne {list(frame.columns)} - INDEX: {frame.index.name}")
        elif isinstance(frame, pd.Series):
            print(f"Frame {i}: Series '{frame.name}' - INDEX: {frame.index.name}")
        else:
            print(f"Frame {i}: Tipo sconosciuto {type(frame)}")

    all_hist_agg = pd.concat(hist_agg_list, ignore_index=True)
    features = features.merge(all_hist_agg, on=['rm_id', 'forecast_start_date'], how='left').fillna(0)

    # Ratios / CV / sparsity
    features['po_to_rec_ratio_30d'] = np.where(features['hist_rec_30d'] > 0,
                                               features['hist_po_30d'] / features['hist_rec_30d'], 0).astype(np.float32)
    features['rec_cv_30d'] = np.where(features['rec_mean_30d'] > 0,
                                         features['rec_std_30d'] / features['rec_mean_30d'], 0).astype(np.float32)
    features['rec_zero_ratio_30d'] = (1.0 - (features.get('rec_count_30d', 0) / 30.0)).astype(np.float32)

    # ---- YoY same-window (strong) ----
    print("  Computing last_year_weight (per-rm)...")
    rec_by_rm = {rm: g.sort_values('date_arrival').reset_index(drop=True)
                 for rm, g in dr.groupby('rm_id')}
    last_year_vals = np.zeros(len(features), dtype=float)
    starts_ly = pd.to_datetime(features["forecast_start_date"]) - pd.Timedelta(days=365)
    ends_ly   = pd.to_datetime(features["forecast_end_date"])   - pd.Timedelta(days=365)

    for rm, rows in features.groupby('rm_id'):
        r = rec_by_rm.get(rm)
        if r is None or r.empty:
            continue
        dates = r['date_arrival'].values.astype('datetime64[D]')
        w = r['net_weight'].values.astype(float)
        c = np.cumsum(w)
        L = np.searchsorted(dates, starts_ly.loc[rows.index].values.astype('datetime64[D]'), side='left')
        R = np.searchsorted(dates, ends_ly.loc[rows.index].values.astype('datetime64[D]'),   side='right') - 1
        s = np.zeros(len(rows), dtype=float)
        for i, (l, rpos) in enumerate(zip(L, R)):
            if rpos >= l and l < len(c):
                s[i] = c[rpos] - (c[l-1] if l > 0 else 0.0)
        last_year_vals[rows.index.values] = s

    features["last_year_weight"] = last_year_vals.astype(np.float32)

    # ---- Seasonal priors (RAM-safe) ----
    print("  Computing rm seasonal priors (global baseline)...")
    dr['month'] = dr['date_arrival'].dt.month
    dr['dow']   = dr['date_arrival'].dt.weekday

    mprior = dr.groupby(["rm_id", "month"])["net_weight"].mean()
    mprior.name = "rm_month_mean_prior"
    
    dprior = dr.groupby(["rm_id", "dow"])["net_weight"].mean()
    dprior.name = "rm_dow_mean_prior"
    
    gprior = dr.groupby("month")["net_weight"].mean()
    gprior.name = "global_month_mean_prior"

    features = features.merge(mprior.reset_index(),
                               left_on=["rm_id", "end_month"], right_on=["rm_id", "month"], how="left").drop(columns=["month"])
    features = features.merge(dprior.reset_index(),
                               left_on=["rm_id", "end_dayofweek"], right_on=["rm_id", "dow"], how="left").drop(columns=["dow"])
    features = features.merge(gprior.reset_index().rename(columns={"month": "end_month"}),
                               on="end_month", how="left")

    features["rm_month_mean_prior"]     = features["rm_month_mean_prior"].fillna(0.0)
    features["rm_dow_mean_prior"]       = features["rm_dow_mean_prior"].fillna(0.0)
    features["global_month_mean_prior"] = features["global_month_mean_prior"].fillna(0.0)
    features["rm_month_vs_global"] = (features["rm_month_mean_prior"] - features["global_month_mean_prior"]).astype(np.float32)

    # ---- Typical load size & expected truck count ----
    load_stats = (dr.groupby(["rm_id", "date_arrival"])["net_weight"].sum()
                      .reset_index()
                      .query("net_weight > 0"))
    
    typical_load = load_stats.groupby("rm_id")["net_weight"].median()
    typical_load.name = "typical_load_rm"
    
    features = features.merge(typical_load.reset_index(), on="rm_id", how="left")
    global_median_load = float(load_stats["net_weight"].median()) if len(load_stats) else 0.0
    features["typical_load_rm"] = features["typical_load_rm"].fillna(global_median_load).astype(np.float32)
    features["expected_truck_count"] = (features["po_in_window"] / (features["typical_load_rm"] + 1e-6)).clip(lower=0).astype(np.float32)

    # ---- (NEW) True lead-time & supplier reliability from raw tables ----
    if (receivals_full is not None) and (purchase_orders_full is not None):
        print("  Computing true lead-time & supplier reliability from raw tables...")
        rf = receivals_full.copy()
        pf = purchase_orders_full.copy()

        # Datetime
        rf["date_arrival"]  = pd.to_datetime(rf["date_arrival"], errors="coerce")
        pf["delivery_date"] = pd.to_datetime(pf["delivery_date"], errors="coerce")

        # Candidate join keys (use what's available)
        join_keys = []
        if "purchase_order_id" in rf.columns and "purchase_order_item_no" in rf.columns:
            join_keys = ["purchase_order_id", "purchase_order_item_no"]
        elif "purchase_order_id" in rf.columns:
            join_keys = ["purchase_order_id"]

        if join_keys:
            jt = rf.merge(
                pf[join_keys + [c for c in ["product_id", "delivery_date", "quantity"] if c in pf.columns]],
                on=join_keys, how="left", suffixes=("", "_po")
            )
            # Guardrail: product id should match if present
            if "product_id_po" in jt.columns and "product_id" in jt.columns:
                mask_pid = jt["product_id_po"].isna() | (jt["product_id_po"] == jt["product_id"])
                jt = jt[mask_pid]

            jt = jt.dropna(subset=["date_arrival", "delivery_date"])
            jt["lead_time_days"] = (jt["date_arrival"] - jt["delivery_date"]).dt.days
            jt = jt[(jt["lead_time_days"] >= -10) & (jt["lead_time_days"] <= 180)]

            # Aggregate per rm_id
            lt_rm = jt.groupby("rm_id")["lead_time_days"].agg(["mean", "std", "median"]).reset_index()
            lt_rm.columns = ["rm_id", "lead_time_mean_rm", "lead_time_std_rm", "lead_time_median_rm"]
            features = features.merge(lt_rm, on="rm_id", how="left")

            # Supplier reliability (dominant supplier per rm_id)
            if "supplier_id" in rf.columns:
                lt_sup = jt.groupby("supplier_id")["lead_time_days"].agg(["mean", "std", "median"]).reset_index()
                lt_sup.columns = ["supplier_id", "sup_lead_mean", "sup_lead_std", "sup_lead_median"]

                dom_sup = (rf.groupby(["rm_id", "supplier_id"])["net_weight"].sum()
                             .reset_index()
                             .sort_values(["rm_id", "net_weight"], ascending=[True, False])
                             .drop_duplicates(["rm_id"])
                             .rename(columns={"supplier_id": "dominant_supplier_id"}))[["rm_id", "dominant_supplier_id"]]

                features = features.merge(dom_sup, on="rm_id", how="left")
                features = features.merge(lt_sup, left_on="dominant_supplier_id", right_on="supplier_id", how="left")
                if "supplier_id" in features.columns:
                    features.drop(columns=["supplier_id"], inplace=True)

                features["supplier_reliability_score"] = (
                    (-features["sup_lead_mean"].fillna(0.0) / 30.0) +
                    (-features["sup_lead_std"].fillna(0.0)  / 15.0)
                ).astype(np.float32)

            for c in ["lead_time_mean_rm", "lead_time_std_rm", "lead_time_median_rm",
                      "sup_lead_mean", "sup_lead_std", "sup_lead_median", "supplier_reliability_score"]:
                if c in features.columns:
                    features[c] = features[c].fillna(0.0).astype(np.float32)
        else:
            # No join keys available: add default 0.0 columns
            for c in ["lead_time_mean_rm", "lead_time_std_rm", "lead_time_median_rm",
                      "sup_lead_mean", "sup_lead_std", "sup_lead_median", "supplier_reliability_score"]:
                features[c] = 0.0 


    # --- Intermittent baseline summed in window (Croston) as feature ---
    # Build once per whole feature set (Jan 1 -> May 31 of same year as each window end)
    win_min = pd.to_datetime(features['forecast_start_date']).min()
    win_max = pd.to_datetime(features['forecast_end_date']).max()
    
    if pd.isna(win_min) or pd.isna(win_max):
        print("Warning: Cannot build intermittent baseline due to NaT dates. Skipping.")
        features['baseline_sum_in_window'] = 0.0
    else:
        baseline_df = build_intermittent_baseline(daily_receivals, win_min, win_max, method="croston", alpha=0.1)
        features['baseline_sum_in_window'] = sum_baseline_over_windows(features, baseline_df).astype(np.float32)


    # ---- Final selection (reduced redundancy) ----
    core_feature_cols = [
        'rm_id',
        'window_length',
        # calendar (clean)
        'end_dayofweek', 'end_days_to_month_end',
        'business_days_in_window', 'weekends_in_window',
        'month_sin', 'month_cos',
        'baseline_sum_in_window',

        # PO window
        'po_in_window', 'daily_avg_po_in_window',
        # historical sums
        'hist_rec_7d', 'hist_rec_30d', 'hist_rec_90d',
        'hist_po_7d',  'hist_po_30d',  'hist_po_90d',
        # rolling stats / trend
        'rec_mean_30d', 'rec_std_30d', 'rec_trend_30d',
        'rec_mean_90d', 'rec_std_90d', 'rec_trend_90d',
        # counts & sparsity
        'rec_count_30d', 'rec_count_90d', 'rec_zero_ratio_30d',
        # ratios
        'po_to_rec_ratio_30d', 'rec_cv_30d',
        # priors & YoY
        'last_year_weight', 'rm_month_mean_prior', 'rm_dow_mean_prior', 'rm_month_vs_global',
        # logistics
        'typical_load_rm', 'expected_truck_count',
        # NEW: lead-time & supplier reliability (if present)
        'lead_time_mean_rm', 'lead_time_std_rm', 'lead_time_median_rm',
        'supplier_reliability_score'
    ]

    final_feature_cols = core_feature_cols + ['index', 'forecast_start_date', 'forecast_end_date']
    for col in core_feature_cols:
        if col not in features.columns:
            features[col] = 0
    return features[final_feature_cols].set_index('index')


def generate_training_data(daily_receivals, daily_po, years_to_use,
                           receivals_full=None, purchase_orders_full=None):
    """
    Generates a training DataFrame by creating cumulative windows
    from historical years, aligned to Jan 1 → May 31 (inclusive).
    """
    print("Generating training data...")
    train_windows = []
    all_rms = daily_receivals['rm_id'].unique()

    for rm in all_rms:
        for year in years_to_use:
            start_date = date(year, 1, 1)
            end_max = date(year, 5, 31)  # inclusive
            max_h = (end_max - start_date).days  # 150 (Jan1..May31 -> 151 days inclusive with +1 later)
            for h in range(0, max_h + 1):  # 0..150
                end_date = start_date + timedelta(days=h)
                train_windows.append({
                    'rm_id': rm,
                    'forecast_start_date': start_date,
                    'forecast_end_date': end_date
                })

    train_df_raw = pd.DataFrame(train_windows)

    # Build features passing raw tables for true lead-time/supplier
    X_train = create_features(train_df_raw, daily_receivals, daily_po,
                               receivals_full=receivals_full,
                               purchase_orders_full=purchase_orders_full)

    # Targets (cumulative receivals in the window)
    print("  Creating training targets (memory-efficient per-rm computation)...")
    y_train = pd.Series(0, index=X_train.index, name='target')
    dr = daily_receivals.copy()
    if dr['date_arrival'].dtype != 'O' and not np.issubdtype(dr['date_arrival'].dtype, np.datetime64):
        dr['date_arrival'] = pd.to_datetime(dr['date_arrival'], errors='coerce')
    rec_by_rm = {rm: grp.sort_values('date_arrival').reset_index(drop=True)
                 for rm, grp in dr.groupby('rm_id')}

    for rm, rows in X_train.reset_index().groupby('rm_id'):
        rows = rows.copy()
        rows_index = rows['index'].values
        rec = rec_by_rm.get(rm)
        if rec is None or rec.empty:
            continue
        rec_dates = pd.to_datetime(rec['date_arrival']).values.astype('datetime64[D]')
        rec_weights = rec['net_weight'].values.astype(float)
        rec_cum = np.cumsum(rec_weights)
        starts = pd.to_datetime(rows['forecast_start_date']).values.astype('datetime64[D]')
        ends = pd.to_datetime(rows['forecast_end_date']).values.astype('datetime64[D]')
        left_pos = np.searchsorted(rec_dates, starts, side='left')
        right_pos = np.searchsorted(rec_dates, ends, side='right') - 1

        sums = np.zeros(len(rows_index), dtype=float)
        for i, (l, r) in enumerate(zip(left_pos, right_pos)):
            if r >= l and r >= 0 and l < len(rec_cum):
                left_cum = rec_cum[l-1] if l > 0 else 0.0
                sums[i] = rec_cum[r] - left_cum
        y_train.loc[rows_index] = sums

    # Safety: ensure no train window leaks into June
    assert (pd.to_datetime(X_train['forecast_end_date']).dt.month <= 5).all(), "Found training windows beyond May!"

    X_train_export = X_train.drop(columns=['forecast_start_date', 'forecast_end_date'], errors='ignore')
    return X_train_export, y_train


def run_data_preparation():
    """
    Loads data, creates features, and saves the final training set.
    """
    data_tuple = load_data()
    if data_tuple[0] is None:
        print("Halting execution due to file loading error.")
        return None

    receivals, purchase_orders, materials, prediction_mapping = data_tuple
    daily_receivals, daily_po = aggregate_daily_weight(receivals, purchase_orders, materials)

   # Build dynamic historical range, EXCLUDING COVID YEARS
    min_year = receivals['date_arrival'].min().year
    max_year = 2024  # Latest full year in the dataset
    
    all_years = list(range(min_year, max_year + 1))
    covid_years = {2020, 2021, 2022}
    years_range = [year for year in all_years if year not in covid_years]
    
    print(f"🎯 Using historical range: {years_range}")
    
    # Generate training data (pass raw tables for true lead-time/supplier)
    X_train, y_train = generate_training_data(
        daily_receivals, daily_po, years_to_use=years_range,
        receivals_full=receivals, purchase_orders_full=purchase_orders
    )

    # Export training dataset
    print("Exporting training dataset...")
    training_data_to_export = X_train.copy()
    training_data_to_export['target'] = y_train

    training_data_filename = 'training_dataset.csv'

    training_data_to_export.to_csv(training_data_filename, index=True)
    print(f"Successfully exported training data to {training_data_filename}")
    return training_data_filename

print("ENHANCED Data preparation functions (Option B) defined.")

## XGBoost Model

In [ ]:
BASE_DIR = PROJECT_ROOT
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# --- Model Configuration ---
ENABLE_HYPERPARAM_TUNING = True
HYPEROPT_N_TRIALS = 20
HYPEROPT_TIMEOUT = None  # seconds or None
RUN_NOTES = "Run con 30+ features, no covid"

def _encode_rm_id_as_codes(X_train_full: pd.DataFrame,
                           X_val: pd.DataFrame,
                           X_test: pd.DataFrame):
    """
    Make rm_id categorical with training categories, then map to stable int codes for XGBoost.
    """
    if 'rm_id' not in X_train_full.columns:
        return X_train_full, X_val, X_test, []

    X_train_full = X_train_full.copy()
    X_train_full['rm_id'] = X_train_full['rm_id'].astype('category')
    rm_categories = X_train_full['rm_id'].cat.categories.tolist()

    def _align(df):
        df = df.copy()
        df['rm_id'] = pd.Categorical(df['rm_id'], categories=rm_categories)
        df['rm_id'] = df['rm_id'].cat.codes.astype('int32')
        return df

    X_val  = _align(X_val)
    X_test = _align(X_test)
    X_train_full['rm_id'] = X_train_full['rm_id'].cat.codes.astype('int32')

    return X_train_full, X_val, X_test, rm_categories

def plot_feature_importance(booster, feature_names, top_n=30):
    """
    Plots the top_n feature importances from a trained XGBoost booster.
    """
    try:
        print("\n--- Feature Importance (Top 30 by 'gain') ---")
        # Get importance by 'gain' (more robust than 'weight')
        importance = booster.get_score(importance_type='gain')
        
        fmap = {f'f{i}': name for i, name in enumerate(feature_names)}
        importance = {fmap.get(k, k): v for k, v in importance.items()}

        imp_df = pd.DataFrame({
            'feature': importance.keys(),
            'importance': importance.values()
        })
        
        imp_df = imp_df.sort_values(by='importance', ascending=False).head(top_n)
        
        # Plot
        plt.figure(figsize=(10, top_n / 2.5))
        sns.barplot(x='importance', y='feature', data=imp_df, palette='viridis')
        plt.title(f'Top {top_n} Feature Importances (Type: Gain)')
        plt.xlabel('Importance (Gain)')
        plt.ylabel('Feature')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Warning: Could not plot feature importance. Error: {e}")


In [ ]:

"""
Main function for the XGBoost model training and prediction pipeline.
"""
# --- 1) Load Training Data ---
training_file = 'training_dataset.csv'
run_data_preparation()
training_df = pd.read_csv(training_file, index_col=0)

sample_weight_series = None
if 'recency_weight' in training_df.columns:
    sample_weight_series = training_df['recency_weight']

# Drop meta cols
drop_meta = [c for c in ['forecast_start_date', 'forecast_end_date', 'recency_weight'] if c in training_df.columns]
X_train = training_df.drop(columns=['target'] + drop_meta, errors='ignore')
y_train = training_df['target']

# --- 2) Build Test Set ---
print("Loading raw data to build test set...")
receivals, purchase_orders, materials, prediction_mapping = load_data()
daily_receivals, daily_po = aggregate_daily_data(receivals, purchase_orders, materials)

test_df_raw = prediction_mapping.set_index('ID')

# IMPORTANT: pass raw tables to create_features (Option B)
X_test = create_features(
    test_df_raw, daily_receivals, daily_po,
    receivals_full=receivals, purchase_orders_full=purchase_orders
)
# Align columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print("Test set created.")

# --- 3) Chronological holdout for validation ---
print("Creating chronological holdout: last 150 rows per rm_id -> validation")
val_idx = X_train.groupby('rm_id').tail(150).index
X_val = X_train.loc[val_idx].copy()
y_val = y_train.loc[val_idx].copy()
train_idx = X_train.index.difference(val_idx)
X_train_full = X_train.loc[train_idx].copy()
y_train_full = y_train.loc[train_idx].copy()
print(f"Training rows: {len(X_train_full)}, Validation rows: {len(X_val)}")

# Encode rm_id
X_train_full, X_val, X_test, rm_categories = _encode_rm_id_as_codes(X_train_full, X_val, X_test)

# Sample weights
if sample_weight_series is None:
    sample_weight_series = pd.Series(1.0, index=X_train.index)
else:
    sample_weight_series = sample_weight_series.reindex(X_train.index).fillna(1.0)
sample_weight_val   = sample_weight_series.loc[val_idx].values
sample_weight_train = sample_weight_series.loc[train_idx].values

# QuantileDMatrix datasets
dtrain = xgb.QuantileDMatrix(X_train_full, y_train_full, weight=sample_weight_train)
dvalid = xgb.QuantileDMatrix(X_val, y_val, weight=sample_weight_val, ref=dtrain)
dtest  = xgb.QuantileDMatrix(X_test, ref=dtrain)

# --- 4) Hyperparameter Tuning (Optuna) ---
base_params = {
    "objective": "reg:quantileerror",
    "quantile_alpha": 0.2,
    "tree_method": "hist",
    "eval_metric": "quantile",
    "seed": SEED,
    "verbosity": 0,
}

best_params = {}
best_iter = None
best_score_for_log = None 
n_trials_for_log = 0

if ENABLE_HYPERPARAM_TUNING:
    print(f"Running Optuna tuning (trials={HYPEROPT_N_TRIALS})...")

    def objective(trial: "optuna.trial.Trial"):
        params = base_params.copy()
        params.update({
            "eta": trial.suggest_float("eta", 1e-3, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 64.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 10.0),
        })
        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=20000,
            evals=[(dtrain, "train"), (dvalid, "valid")],
            early_stopping_rounds=200,
            verbose_eval=False,
        )
        trial.set_user_attr("best_iteration", booster.best_iteration)
        trial.set_user_attr("best_score", booster.best_score)
        return float(booster.best_score)

    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=HYPEROPT_N_TRIALS, timeout=HYPEROPT_TIMEOUT)
    best_params = study.best_params
    best_iter = int(study.best_trial.user_attrs.get("best_iteration", 1000))
    print(f"Best params: {best_params}")
    print(f"Best iteration: {best_iter}")

    # saves score for logging 
    best_score_for_log = study.best_value
    n_trials_for_log = len(study.trials)
    print(f"Best params: {best_params}")
    print(f"Best iteration: {best_iter}")
    print(f"Best validation score (pinball loss): {best_score_for_log}") 

else:
    print("Tuning disabled. Using reasonable defaults and estimating best iteration...")
    best_params = {
        "eta": 0.05, "max_depth": 6, "min_child_weight": 1.0,
        "subsample": 0.9, "colsample_bytree": 0.9,
        "reg_alpha": 0.0, "reg_lambda": 1.0, "gamma": 0.0,
    }
    params = base_params.copy(); params.update(best_params)
    tmp_booster = xgb.train(
        params=params, dtrain=dtrain, num_boost_round=20000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=200, verbose_eval=False,
    )
    best_iter = tmp_booster.best_iteration
    print(f"Estimated best iteration: {best_iter}")
    # saves score for logging
    best_score_for_log = tmp_booster.best_score
    n_trials_for_log = 0 
    print(f"Estimated best iteration: {best_iter}")
    print(f"Validation score (pinball loss): {best_score_for_log}")


In [ ]:

# --- 5) Train final model on train+val ---
print("Retraining final model on train+val at best iteration...")
X_combined = pd.concat([X_train_full, X_val], axis=0)
y_combined = pd.concat([y_train_full, y_val], axis=0)
sw_combined = pd.concat(
    [pd.Series(sample_weight_train, index=X_train_full.index),
        pd.Series(sample_weight_val, index=X_val.index)],
    axis=0
).reindex(X_combined.index).fillna(1.0).values

dcomb = xgb.QuantileDMatrix(X_combined, y_combined, weight=sw_combined)
final_params = base_params.copy()
final_params.update(best_params)

booster_final = xgb.train(
    params=final_params,
    dtrain=dcomb,
    num_boost_round=(best_iter + 1 if best_iter is not None else 1000),
    verbose_eval=False,
)

# Plot feature importance
plot_feature_importance(booster_final, X_train_full.columns, top_n=30)

# Log score to CSV
print("Logging score to score_log.csv...")
log_file_path = BASE_DIR / 'data/data_modified/score_log.csv'
log_entry = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_validation_score": best_score_for_log,
    "best_iteration": best_iter,
    "n_trials": n_trials_for_log,
    "notes": RUN_NOTES,
    "features_used": "|".join(X_train.columns.tolist()) 
}
log_df = pd.DataFrame([log_entry])

try:
    if not log_file_path.exists():
        log_df.to_csv(log_file_path, index=False)
    else:
        log_df.to_csv(log_file_path, mode='a', header=False, index=False)
    print(f"Score successfully logged to {log_file_path}")
except Exception as e:
    print(f"Error logging score: {e}")

# Save model and metadata
model_out_dir = BASE_DIR / 'outputs' / 'model'
os.makedirs(model_out_dir, exist_ok=True)
model_file = model_out_dir / 'best_xgb_model.json'
meta_file  = model_out_dir / 'best_xgb_model_meta.json'
booster_final.save_model(str(model_file))

meta = {
    'feature_columns': X_train.columns.tolist(),
    'categorical_rm_id_categories': rm_categories,
    'quantile_alpha': 0.2,
    'library': 'xgboost',
    'best_iteration': int(best_iter) if best_iter is not None else None,
    'tuning_used': bool(ENABLE_HYPERPARAM_TUNING),
    'best_params': best_params,
    'best_validation_score': best_score_for_log
}
with open(meta_file, 'w', encoding='utf-8') as fh:
    json.dump(meta, fh, indent=2)
print(f"Saved model to {model_file} and metadata to {meta_file}")


In [ ]:

# --- 6) Prediction & Submission ---
print("Generating predictions...")
preds = booster_final.predict(dtest)
preds = np.asarray(preds, dtype=float)
preds[preds < 10] = 0.0  # non-negative and conservative

submission = pd.DataFrame({'ID': X_test.index, 'predicted_weight': preds})
mapping_small = prediction_mapping[['ID', 'rm_id', 'forecast_end_date']].copy()
mapping_small['forecast_end_date'] = pd.to_datetime(mapping_small['forecast_end_date'])

merged_sub = submission.merge(mapping_small, on='ID', how='left').sort_values(['rm_id', 'forecast_end_date'])
# Per-rm_id cummax: predictions are cumulative by window end
merged_sub['predicted_weight'] = merged_sub.groupby('rm_id')['predicted_weight'].cummax()
submission = merged_sub.sort_values('ID')[['ID', 'predicted_weight']].reset_index(drop=True)

submission_filename = 'submission_xgb.csv'
submission.to_csv(submission_filename, index=False)
print(f"Saved submission to {submission_filename}")
print("--- XGBoost Pipeline Completed! ---")
